# Thực nghiệm Trans2Vec trên bộ dữ liệu `food`

**Bài báo:** *Trans2Vec: Learning Transaction Embedding via Items and Frequent Itemsets* (PAKDD 2018).

**Mục tiêu:** Chạy thực tế mô hình Trans2Vec dựa trên PV-DBOW (Doc2Vec) kết hợp items và frequent itemsets, phân loại bằng LinearSVC (3200 train / 800 test, 10 runs).

In [4]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# Tìm thư mục dự án office-lunch
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline" / "data" / "food.txt").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy pipeline/data/food.txt!")

DATA = ROOT / "pipeline" / "data"
OUT = ROOT / "pipeline" / "out"
OUT.mkdir(exist_ok=True)

DIM = 128        # Số chiều vector
EPOCHS = 50      # Số vòng train PV-DBOW
N_TRAIN = 3200   # 3200 giao dịch train, 800 test
N_RUNS = 10      # Chạy 10 lần lấy trung bình

print("Thư mục dự án:", ROOT)

Thư mục dự án: /Users/huyhd/Huy Form/University/CSDL/Project/office-lunch


## 1. Tải dữ liệu thực tế

In [5]:
def load_data(path):
    X, y = [], []
    for line in open(path, encoding="utf-8"):
        parts = line.rstrip().split("\t")
        if len(parts) == 2:
            label, content = parts
            if content.strip():
                y.append(label)
                X.append(content.split(" "))
    return X, np.array(y)

items_X, y = load_data(DATA / "food.txt")
print(f"Tổng số giao dịch tải được: {len(items_X)}")
print(f"Phân bố nhãn: yes={(y=='yes').sum()}, no={(y=='no').sum()}")

Tổng số giao dịch tải được: 4000
Phân bố nhãn: yes=2000, no=2000


## 2. Huấn luyện thực tế Doc2Vec (PV-DBOW) & Đánh giá LinearSVC

In [6]:
documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(items_X)]

print("Đang huấn luyện mô hình Doc2Vec (PV-DBOW) thực tế...")
model = Doc2Vec(documents, vector_size=DIM, window=10, min_count=0, workers=4, epochs=EPOCHS, dm=0)

X_vectors = np.array([model.dv[i] for i in range(len(items_X))])
X_train, X_test = X_vectors[:N_TRAIN], X_vectors[N_TRAIN:]
y_train, y_test = y[:N_TRAIN], y[N_TRAIN:]

accuracies, macro_f1s = [], []
for run in range(N_RUNS):
    clf = LinearSVC(random_state=42 + run, max_iter=3000)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    accuracies.append(accuracy_score(y_test, y_pred) * 100)
    macro_f1s.append(f1_score(y_test, y_pred, average='macro') * 100)

print("="*50)
print(f"[TRANS2VEC] Kết quả thực tế trên Foodmart (10 runs):")
print(f"- Macro-F1 : {np.mean(macro_f1s):.2f}% ± {np.std(macro_f1s):.2f}%")
print("="*50)

Đang huấn luyện mô hình Doc2Vec (PV-DBOW) thực tế...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


[TRANS2VEC] Kết quả thực tế trên Foodmart (10 runs):
- Macro-F1 : 67.91% ± 0.00%
